In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')


In [ ]:
from google.colab import drive

# 드라이브 마운트
drive.mount('/content/drive')

# 1. 파일이 저장된 정확한 경로 (본인의 경로에 맞게 수정하세요)
base_path = '/content/drive/MyDrive/ColabNotebooks/'

Mounted at /content/drive


In [ ]:
training_data = pd.read_csv(f'{base_path}albumin_training_data.csv')

print(f"\n✅ 데이터 로드 완료: {len(training_data):,}건")
print("\n📊 데이터 미리보기:")
print(training_data.head())

print("\n📊 데이터 정보:")
print(training_data.info())

print("\n📊 기술통계:")
print(training_data.describe())

print("\n📊 결측치 확인:")
print(training_data.isnull().sum())


✅ 데이터 로드 완료: 506건

📊 데이터 미리보기:
   AGE  GENDER  CKD  DIABETES  CCI  INITIAL_ALBUMIN  PROTEIN_INTAKE  \
0   60       1    0         0    2              3.8             0.0   
1   64       0    0         0    0              3.8             0.0   
2   71       1    0         0    0              2.6             0.0   
3   65       1    0         1    3              3.9             0.0   
4   67       0    0         0    0              4.1             0.0   

   DAYS_BETWEEN  ALBUMIN_CHANGE  FINAL_ALBUMIN  
0            29          -0.775          3.025  
1            25          -0.250          3.550  
2            26           0.200          2.800  
3            27          -0.400          3.500  
4            25          -0.900          3.200  

📊 데이터 정보:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   AGE              506 non-null    i

In [ ]:
training_data.columns.tolist()

['AGE',
 'GENDER',
 'CKD',
 'DIABETES',
 'CCI',
 'INITIAL_ALBUMIN',
 'PROTEIN_INTAKE',
 'DAYS_BETWEEN',
 'ALBUMIN_CHANGE',
 'FINAL_ALBUMIN']

In [ ]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3-2. 데이터 전처리
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "=" * 80)
print("데이터 전처리")
print("=" * 80)

# 필수 컬럼 확인
required_cols = ['AGE', 'GENDER', 'CKD', 'DIABETES','INITIAL_ALBUMIN', 'PROTEIN_INTAKE', 'ALBUMIN_CHANGE']

missing_cols = set(required_cols) - set(training_data.columns)
if missing_cols:
    print(f"❌ 필수 컬럼 누락: {missing_cols}")
    # 누락된 컬럼이 있으면 0으로 채우기
    for col in missing_cols:
        training_data[col] = 0
        print(f"   {col} 컬럼을 0으로 생성")

# 결측치 제거
print(f"\n원본 데이터: {len(training_data):,}건")
data_clean = training_data[required_cols].dropna()
print(f"결측치 제거 후: {len(data_clean):,}건 (제거: {len(training_data) - len(data_clean):,}건)")

# 이상치 제거
print("\n이상치 제거:")
print(f"  - Albumin 변화량이 ±3.0 g/dL를 초과하는 케이스")
print(f"  - 단백질 섭취량이 200g/day를 초과하는 케이스")
print(f"  - 초기 Albumin이 1.5~5.0 범위를 벗어나는 케이스")

before_outlier = len(data_clean)
data_clean = data_clean[
    (data_clean['ALBUMIN_CHANGE'].abs() <= 2.0) &
    (data_clean['INITIAL_ALBUMIN'].between(1.0, 5.5))
]

data_clean['PROTEIN_INTAKE'] = data_clean['PROTEIN_INTAKE'].clip(upper=300)

print(f"이상치 제거 후: {len(data_clean):,}건 (제거: {before_outlier - len(data_clean):,}건)")

# 최종 데이터 분포 확인
print("\n📊 최종 데이터 분포:")
print("\nAlbumin 변화량 분포:")
print(data_clean['ALBUMIN_CHANGE'].describe())

print("\n단백질 섭취량 분포:")
print(data_clean['PROTEIN_INTAKE'].describe())

print("\n질환별 분포:")
print(f"  CKD 환자: {data_clean['CKD'].sum():,}건 ({data_clean['CKD'].sum()/len(data_clean)*100:.1f}%)")
print(f"  당뇨 환자: {data_clean['DIABETES'].sum():,}건 ({data_clean['DIABETES'].sum()/len(data_clean)*100:.1f}%)")




데이터 전처리

원본 데이터: 506건
결측치 제거 후: 506건 (제거: 0건)

이상치 제거:
  - Albumin 변화량이 ±3.0 g/dL를 초과하는 케이스
  - 단백질 섭취량이 200g/day를 초과하는 케이스
  - 초기 Albumin이 1.5~5.0 범위를 벗어나는 케이스
이상치 제거 후: 504건 (제거: 2건)

📊 최종 데이터 분포:

Albumin 변화량 분포:
count    504.000000
mean      -0.252247
std        0.515671
min       -1.933333
25%       -0.571250
50%       -0.236667
75%        0.082197
max        1.500000
Name: ALBUMIN_CHANGE, dtype: float64

단백질 섭취량 분포:
count    504.000000
mean       0.178571
std        2.849411
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       50.000002
Name: PROTEIN_INTAKE, dtype: float64

질환별 분포:
  CKD 환자: 83건 (16.5%)
  당뇨 환자: 120건 (23.8%)


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3-3. Feature/Target 분리
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "=" * 80)
print("Feature/Target 분리")
print("=" * 80)

# Feature 정의
feature_names = ['AGE', 'GENDER', 'CKD', 'DIABETES', 'INITIAL_ALBUMIN', 'PROTEIN_INTAKE']

# CCI가 있으면 추가
if 'CCI' in data_clean.columns:
    feature_names.append('CCI')
    print("✅ Charlson Comorbidity Index 포함")

print(f"\n사용 Features ({len(feature_names)}개):")
for i, feat in enumerate(feature_names, 1):
    print(f"  {i}. {feat}")

# X, y 분리
X = data_clean[feature_names].copy()
y = data_clean['ALBUMIN_CHANGE'].copy()

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")


Feature/Target 분리

사용 Features (6개):
  1. AGE
  2. GENDER
  3. CKD
  4. DIABETES
  5. INITIAL_ALBUMIN
  6. PROTEIN_INTAKE

X shape: (504, 6)
y shape: (504,)


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3-4. Train/Test 분할
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "=" * 80)
print("Train/Test 분할 (80:20)")
print("=" * 80)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print(f"\n📈 Train set: {len(X_train):,}건")
print(f"📈 Test set: {len(X_test):,}건")

print("\nTrain set - Target 분포:")
print(y_train.describe())

print("\nTest set - Target 분포:")
print(y_test.describe())


Train/Test 분할 (80:20)

📈 Train set: 403건
📈 Test set: 101건

Train set - Target 분포:
count    403.000000
mean      -0.270778
std        0.522577
min       -1.933333
25%       -0.600000
50%       -0.250000
75%        0.078030
max        1.400000
Name: ALBUMIN_CHANGE, dtype: float64

Test set - Target 분포:
count    101.000000
mean      -0.178308
std        0.482540
min       -1.150000
25%       -0.480000
50%       -0.166667
75%        0.081818
max        1.500000
Name: ALBUMIN_CHANGE, dtype: float64


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3-5. 모델 정의 및 학습
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "=" * 80)
print("Gradient Boosting 모델 학습")
print("=" * 80)

# 모델 초기화
model = GradientBoostingRegressor(
    n_estimators=200,          # 트리 개수
    learning_rate=0.05,        # 학습률
    max_depth=4,               # 트리 깊이
    min_samples_split=20,      # 노드 분할 최소 샘플 수
    min_samples_leaf=10,       # 리프 노드 최소 샘플 수
    subsample=0.8,             # 서브샘플링 비율
    random_state=42,
    verbose=1                   # 학습 과정 출력
)

print("\n모델 하이퍼파라미터:")
print(model.get_params())

print("\n⏳ 모델 학습 중...")
model.fit(X_train, y_train)
print("✅ 학습 완료!")


Gradient Boosting 모델 학습

모델 하이퍼파라미터:
{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.05, 'loss': 'squared_error', 'max_depth': 4, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 10, 'min_samples_split': 20, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 200, 'n_iter_no_change': None, 'random_state': 42, 'subsample': 0.8, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 1, 'warm_start': False}

⏳ 모델 학습 중...
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           0.2577           0.0066            2.14s
         2           0.2475           0.0018            1.21s
         3           0.2406           0.0116            0.90s
         4           0.2318           0.0017            0.74s
         5           0.2349           0.0456            0.64s
         6           0.2281           0.0052            0.59s
         7           0.2240           0.0103        

In [ ]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3-6. 모델 평가
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "=" * 80)
print("모델 성능 평가")
print("=" * 80)

# 예측
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Train 성능
train_r2 = r2_score(y_train, y_train_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))

# Test 성능
test_r2 = r2_score(y_test, y_test_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print("\n📊 Train Set 성능:")
print(f"   R² Score: {train_r2:.4f}")
print(f"   MAE:      {train_mae:.4f} g/dL")
print(f"   RMSE:     {train_rmse:.4f} g/dL")

print("\n📊 Test Set 성능:")
print(f"   R² Score: {test_r2:.4f}")
print(f"   MAE:      {test_mae:.4f} g/dL")
print(f"   RMSE:     {test_rmse:.4f} g/dL")

# Overfitting 체크
if train_r2 - test_r2 > 0.1:
    print("\n⚠️  과적합 가능성 있음 (Train R² - Test R² > 0.1)")
else:
    print("\n✅ 과적합 없음")




모델 성능 평가

📊 Train Set 성능:
   R² Score: 0.5520
   MAE:      0.2741 g/dL
   RMSE:     0.3493 g/dL

📊 Test Set 성능:
   R² Score: 0.1352
   MAE:      0.3510 g/dL
   RMSE:     0.4465 g/dL

⚠️  과적합 가능성 있음 (Train R² - Test R² > 0.1)


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 전략 1: 정규화 (Regularization) - 하이퍼파라미터 조정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "=" * 80)
print("전략 1: 정규화 강화 (더 보수적인 하이퍼파라미터)")
print("=" * 80)

# 과적합 방지를 위한 설정
model_regularized = GradientBoostingRegressor(
    n_estimators=100,          # 트리 개수 감소 (200 → 100)
    learning_rate=0.01,        # 학습률 대폭 감소 (0.05 → 0.01)
    max_depth=3,               # 트리 깊이 감소 (4 → 3)
    min_samples_split=50,      # 노드 분할 기준 강화 (20 → 50)
    min_samples_leaf=25,       # 리프 노드 기준 강화 (10 → 25)
    subsample=0.7,             # 서브샘플링 강화 (0.8 → 0.7)
    max_features='sqrt',       # Feature 서브샘플링 추가
    random_state=42
)

print("\n⏳ 학습 중...")
model_regularized.fit(X_train, y_train)

y_train_pred = model_regularized.predict(X_train)
y_test_pred = model_regularized.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print(f"\n📊 정규화 모델 성능:")
print(f"   Train R²: {train_r2:.4f}, MAE: {train_mae:.4f}")
print(f"   Test R²:  {test_r2:.4f}, MAE: {test_mae:.4f}")
print(f"   R² 차이:  {train_r2 - test_r2:.4f}")

if train_r2 - test_r2 < 0.1:
    print("   ✅ 과적합 개선!")


전략 1: 정규화 강화 (더 보수적인 하이퍼파라미터)

⏳ 학습 중...

📊 정규화 모델 성능:
   Train R²: 0.2196, MAE: 0.3605
   Test R²:  0.1464, MAE: 0.3329
   R² 차이:  0.0732
   ✅ 과적합 개선!


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3-7. Cross-Validation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "=" * 80)
print("5-Fold Cross-Validation")
print("=" * 80)

cv_scores = cross_val_score(
    model, X_train, y_train,
    cv=5,
    scoring='r2',
    verbose=1
)

print(f"\n각 Fold별 R² Score:")
for i, score in enumerate(cv_scores, 1):
    print(f"   Fold {i}: {score:.4f}")

print(f"\n평균 R²: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")


5-Fold Cross-Validation
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           0.2464           0.0065            0.38s
         2           0.2335          -0.0148            0.34s
         3           0.2297           0.0220            0.34s
         4           0.2206          -0.0052            0.32s
         5           0.2010          -0.0471            0.29s
         6           0.2220           0.1111            0.28s
         7           0.1930          -0.0886            0.26s
         8           0.1921           0.0207            0.25s
         9           0.1866          -0.0039            0.25s
        10           0.1853           0.0139            0.24s
        20           0.1510          -0.0344            0.21s
        30           0.1453          -0.0498            0.19s
        40           0.1422          -0.0568            0.18s
        50           0.1333          -0.0442            0.16s
        60           0.1404           0.0414

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    1.1s finished


In [ ]:
['AGE',
 'GENDER',
 'CKD',
 'DIABETES',
 'CCI',
 'INITIAL_ALBUMIN',
 'PROTEIN_INTAKE',
 'DAYS_BETWEEN',
 'ALBUMIN_CHANGE',
 'FINAL_ALBUMIN']

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3-8. Feature Importance
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "=" * 80)
print("Feature Importance")
print("=" * 80)

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n중요도 순위:")
for idx, row in feature_importance.iterrows():
    bar = '█' * int(row['importance'] * 100)
print(f"   {row['feature']:20s}: {row['importance']:.4f} {bar}")


Feature Importance

중요도 순위:
   INITIAL_ALBUMIN     : 0.6687 ██████████████████████████████████████████████████████████████████
   AGE                 : 0.2299 ██████████████████████
   GENDER              : 0.0563 █████
   DIABETES            : 0.0267 ██
   CKD                 : 0.0185 █
   PROTEIN_INTAKE      : 0.0000 


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3-9. 예측 결과 분석
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "=" * 80)
print("예측 결과 상세 분석")
print("=" * 80)

# 잔차 계산
train_residuals = y_train - y_train_pred
test_residuals = y_test - y_test_pred

print("\n잔차(Residual) 분석:")
print(f"   Train 평균 잔차: {train_residuals.mean():.4f} g/dL")
print(f"   Test 평균 잔차:  {test_residuals.mean():.4f} g/dL")

# 예측 범위 확인
print("\n예측값 범위:")
print(f"   Train: [{y_train_pred.min():.2f}, {y_train_pred.max():.2f}] g/dL")
print(f"   Test:  [{y_test_pred.min():.2f}, {y_test_pred.max():.2f}] g/dL")

# 실제값 범위
print("\n실제값 범위:")
print(f"   Train: [{y_train.min():.2f}, {y_train.max():.2f}] g/dL")
print(f"   Test:  [{y_test.min():.2f}, {y_test.max():.2f}] g/dL")


예측 결과 상세 분석

잔차(Residual) 분석:
   Train 평균 잔차: 0.0003 g/dL
   Test 평균 잔차:  0.0668 g/dL

예측값 범위:
   Train: [-0.53, -0.06] g/dL
   Test:  [-0.52, -0.07] g/dL

실제값 범위:
   Train: [-1.93, 1.40] g/dL
   Test:  [-1.15, 1.50] g/dL


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3-10. 모델 저장
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "=" * 80)
print("모델 저장")
print("=" * 80)

# 저장 디렉토리 생성
os.makedirs('models', exist_ok=True)

# 모델과 메타데이터 저장
model_data = {
    'model': model,
    'feature_names': feature_names,
    'train_r2': train_r2,
    'test_r2': test_r2,
    'train_mae': train_mae,
    'test_mae': test_mae,
    'cv_mean': cv_scores.mean(),
    'cv_std': cv_scores.std(),
    'feature_importance': feature_importance.to_dict()
}

model_path = f'{base_path}albumin_predictor.pkl'
joblib.dump(model_data, model_path)

print(f"✅ 모델 저장 완료: {model_path}")
print(f"   파일 크기: {os.path.getsize(model_path) / 1024:.2f} KB")


모델 저장
✅ 모델 저장 완료: /content/drive/MyDrive/ColabNotebooks/albumin_predictor.pkl
   파일 크기: 318.06 KB
